In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_PATH = (
    PROJECT_ROOT / "data" / "processed" / "transactions_cleaned.csv.gz"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input exists:", CLEANED_PATH.exists())
print("Input:", CLEANED_PATH)

Input exists: True
Input: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\data\processed\transactions_cleaned.csv.gz


In [2]:
if not CLEANED_PATH.exists():
    raise FileNotFoundError(
        "Run Notebook 02 first and verify that transactions_cleaned.csv.gz exists."
    )

df = pd.read_csv(CLEANED_PATH, low_memory=False)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

assert "isFraud" in df.columns, "Target column is missing."
assert df["isFraud"].isin([0, 1]).all(), "Unexpected target labels."

Shape: (6362620, 14)
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'hour_of_day', 'origin_emptied', 'destination_balance_zero']


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,hour_of_day,origin_emptied,destination_balance_zero
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,1,False,True
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,1,False,True
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,1,True,True
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,1,True,True
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,1,False,True


In [3]:
df["origin_balance_error"] = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
)

df["destination_balance_error"] = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
)

# Absolute errors make the size of the discrepancy easier to compare.
df["origin_balance_error_abs"] = df["origin_balance_error"].abs()
df["destination_balance_error_abs"] = (
    df["destination_balance_error"].abs()
)

display(
    df[
        [
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "origin_balance_error",
            "oldbalanceDest",
            "newbalanceDest",
            "destination_balance_error",
        ]
    ].head(10)
)

print("Origin balance error summary:")
display(df["origin_balance_error"].describe())

print("Destination balance error summary:")
display(df["destination_balance_error"].describe())

,amount,oldbalanceOrg,newbalanceOrig,origin_balance_error,oldbalanceDest,newbalanceDest,destination_balance_error
0,9839.64,170136.00,160296.36,0.000000e+00,0.0,0.00,9839.64
1,1864.28,21249.00,19384.72,0.000000e+00,0.0,0.00,1864.28
2,181.00,181.00,0.00,0.000000e+00,0.0,0.00,181.00
3,181.00,181.00,0.00,0.000000e+00,21182.0,0.00,21363.00
4,11668.14,41554.00,29885.86,0.000000e+00,0.0,0.00,11668.14
5,7817.71,53860.00,46042.29,0.000000e+00,0.0,0.00,7817.71
6,7107.77,183195.00,176087.23,0.000000e+00,0.0,0.00,7107.77
7,7861.64,176087.23,168225.59,0.000000e+00,0.0,0.00,7861.64
8,4024.36,2671.00,0.00,-1.353360e+03,0.0,0.00,4024.36
9,5337.77,41720.00,36382.23,-7.275958e-12,41898.0,40348.79,6886.98


Origin balance error summary:


count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
Name: origin_balance_error, dtype: float64

Destination balance error summary:


count    6.362620e+06
mean     5.556717e+04
std      4.415288e+05
min     -7.588573e+07
25%      0.000000e+00
50%      3.500490e+03
75%      2.935305e+04
max      1.319123e+07
Name: destination_balance_error, dtype: float64

In [4]:
df["origin_emptied"] = (
    df["newbalanceOrig"].eq(0)
    & df["oldbalanceOrg"].gt(0)
).astype("int8")

df["destination_balance_zero"] = (
    df["newbalanceDest"].eq(0)
).astype("int8")

print("Origin emptied:")
display(df["origin_emptied"].value_counts())

print("\nDestination balance zero:")
display(df["destination_balance_zero"].value_counts())

print("\nFraud rate by origin emptied flag:")
display(
    df.groupby("origin_emptied")["isFraud"]
      .agg(["count", "sum", "mean"])
)

Origin emptied:


origin_emptied
0    4842039
1    1520581
Name: count, dtype: int64


Destination balance zero:


destination_balance_zero
0    3923187
1    2439433
Name: count, dtype: int64


Fraud rate by origin emptied flag:


,count,sum,mean
origin_emptied,,,
0,4842039,201,0.000042
1,1520581,8012,0.005269


In [5]:
df["hour_of_day"] = (df["step"] % 24).astype("int8")

df["night_flag"] = (
    (df["hour_of_day"] >= 22)
    | (df["hour_of_day"] <= 5)
).astype("int8")

display(
    df[["step", "hour_of_day", "night_flag", "isFraud"]].head(15)
)

print("Transactions by simulated hour:")
display(df["hour_of_day"].value_counts().sort_index())

print("\nFraud rate by night flag:")
display(
    df.groupby("night_flag")["isFraud"]
      .agg(["count", "sum", "mean"])
)

,step,hour_of_day,night_flag,isFraud
0,1,1,1,0
1,1,1,1,0
2,1,1,1,1
3,1,1,1,1
4,1,1,1,0
5,1,1,1,0
6,1,1,1,0
7,1,1,1,0
8,1,1,1,0
9,1,1,1,0


Transactions by simulated hour:


hour_of_day
0      71587
1      27111
2       9018
3       2007
4       1241
5       1641
6       3420
7       8988
8      26915
9     283518
10    425729
11    445992
12    483418
13    468474
14    439653
15    416686
16    441612
17    439941
18    580509
19    647814
20    553728
21    247806
22    194555
23    141257
Name: count, dtype: int64


Fraud rate by night flag:


,count,sum,mean
night_flag,,,
0,5914203,5543,0.000937
1,448417,2670,0.005954


In [6]:
EPSILON = 1.0

df["amount_to_origin_balance_ratio"] = (
    df["amount"] / df["oldbalanceOrg"].clip(lower=EPSILON)
)

# Track transactions where the origin had no recorded opening balance.
df["origin_balance_was_zero"] = (
    df["oldbalanceOrg"].eq(0)
).astype("int8")

print("Amount-to-balance ratio summary:")
display(df["amount_to_origin_balance_ratio"].describe())

print("Infinite ratios:",
      np.isinf(df["amount_to_origin_balance_ratio"]).sum())

print("Missing ratios:",
      df["amount_to_origin_balance_ratio"].isna().sum())

Amount-to-balance ratio summary:


count    6.362620e+06
mean     7.068329e+04
std      5.084277e+05
min      0.000000e+00
25%      2.344085e-01
50%      6.454800e+00
75%      1.229609e+04
max      9.244552e+07
Name: amount_to_origin_balance_ratio, dtype: float64

Infinite ratios: 0
Missing ratios: 0


In [7]:
# Count transactions for each origin account at each simulation step.
origin_step_counts = (
    df.groupby(["nameOrig", "step"], sort=False, observed=True)
      .size()
      .rename("transactions_in_step")
      .reset_index()
)

# Cumulative transaction count for each account, minus the count
# at the current step, gives transactions from earlier steps only.
origin_step_counts["origin_txn_count_prior_steps"] = (
    origin_step_counts
    .groupby("nameOrig", sort=False, observed=True)["transactions_in_step"]
    .cumsum()
    - origin_step_counts["transactions_in_step"]
)

# Merge the past-only count back onto each transaction.
df = df.merge(
    origin_step_counts[
        ["nameOrig", "step", "origin_txn_count_prior_steps"]
    ],
    on=["nameOrig", "step"],
    how="left",
    validate="many_to_one",
    sort=False,
)

df["origin_txn_count_prior_steps"] = (
    df["origin_txn_count_prior_steps"].fillna(0).astype("int32")
)

print("Origin velocity feature created.")
display(
    df[
        ["nameOrig", "step", "origin_txn_count_prior_steps", "isFraud"]
    ].head(15)
)

print("Velocity feature summary:")
display(df["origin_txn_count_prior_steps"].describe())

Origin velocity feature created.


,nameOrig,step,origin_txn_count_prior_steps,isFraud
0,C1231006815,1,0,0
1,C1666544295,1,0,0
2,C1305486145,1,0,1
3,C840083671,1,0,1
4,C2048537720,1,0,0
5,C90045638,1,0,0
6,C154988899,1,0,0
7,C1912850431,1,0,0
8,C1265012928,1,0,0
9,C712410124,1,0,0


Velocity feature summary:


count    6.362620e+06
mean     1.458676e-03
std      3.822647e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.000000e+00
Name: origin_txn_count_prior_steps, dtype: float64

In [8]:
type_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.int8,
)

type_encoded_array = type_encoder.fit_transform(df[["type"]])

type_encoded = pd.DataFrame(
    type_encoded_array,
    columns=[
        f"type_{category}"
        for category in type_encoder.categories_[0]
    ],
    index=df.index,
)

print("Categories:", type_encoder.categories_[0].tolist())
display(type_encoded.head())

print("One-hot columns:")
print(type_encoded.columns.tolist())

Categories: ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']


,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,0,0,0,1,0
1,0,0,0,1,0
2,0,0,0,0,1
3,0,1,0,0,0
4,0,0,0,1,0


One-hot columns:
['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [9]:
engineered_features = [
    "origin_balance_error",
    "destination_balance_error",
    "origin_balance_error_abs",
    "destination_balance_error_abs",
    "origin_emptied",
    "destination_balance_zero",
    "hour_of_day",
    "night_flag",
    "amount_to_origin_balance_ratio",
    "origin_balance_was_zero",
    "origin_txn_count_prior_steps",
]

print("Engineered features:", len(engineered_features))
display(df[engineered_features].describe().T)

print("\nMissing values in engineered features:")
display(df[engineered_features].isna().sum())

print("\nInfinite values in engineered features:")
display(
    pd.Series(
        np.isinf(df[engineered_features].select_dtypes(include=np.number)).sum(),
        name="infinite_count",
    )
)

Engineered features: 11


,count,mean,std,min,25%,50%,75%,max
origin_balance_error,6362620.0,-201092.468053,606650.460140,-92445516.64,-249641.087500,-68677.2550,-2954.2300,0.01
destination_balance_error,6362620.0,55567.171867,441528.768827,-75885725.63,0.000000,3500.4900,29353.0450,13191233.98
origin_balance_error_abs,6362620.0,201092.468248,606650.460075,0.00,2954.230000,68677.2550,249641.0875,92445516.64
destination_balance_error_abs,6362620.0,93599.071898,435056.981486,0.00,0.000000,5123.6200,43421.3300,75885725.63
origin_emptied,6362620.0,0.238987,0.426465,0.00,0.000000,0.0000,0.0000,1.00
destination_balance_zero,6362620.0,0.383401,0.486215,0.00,0.000000,0.0000,1.0000,1.00
hour_of_day,6362620.0,15.321454,4.321799,0.00,12.000000,16.0000,19.0000,23.00
night_flag,6362620.0,0.070477,0.255949,0.00,0.000000,0.0000,0.0000,1.00
amount_to_origin_balance_ratio,6362620.0,70683.293860,508427.668787,0.00,0.234408,6.4548,12296.0925,92445516.64
origin_balance_was_zero,6362620.0,0.330438,0.470371,0.00,0.000000,0.0000,1.0000,1.00



Missing values in engineered features:


origin_balance_error              0
destination_balance_error         0
origin_balance_error_abs          0
destination_balance_error_abs     0
origin_emptied                    0
destination_balance_zero          0
hour_of_day                       0
night_flag                        0
amount_to_origin_balance_ratio    0
origin_balance_was_zero           0
origin_txn_count_prior_steps      0
dtype: int64


Infinite values in engineered features:


origin_balance_error              0
destination_balance_error         0
origin_balance_error_abs          0
destination_balance_error_abs     0
origin_emptied                    0
destination_balance_zero          0
hour_of_day                       0
night_flag                        0
amount_to_origin_balance_ratio    0
origin_balance_was_zero           0
origin_txn_count_prior_steps      0
Name: infinite_count, dtype: int64

In [10]:
FEATURES_PATH = PROCESSED_DIR / "transactions_features.csv.gz"

df.to_csv(
    FEATURES_PATH,
    index=False,
    compression="gzip",
)

feature_report = pd.DataFrame({
    "feature": engineered_features,
    "dtype": df[engineered_features].dtypes.astype(str).values,
    "missing_count": df[engineered_features].isna().sum().values,
})

feature_report.to_csv(
    OUTPUT_DIR / "engineered_feature_report.csv",
    index=False,
)

print("Feature dataset saved:", FEATURES_PATH)
print("Feature report saved:", OUTPUT_DIR / "engineered_feature_report.csv")
print("Final shape:", df.shape)
print("Compressed file size (MB):",
      round(FEATURES_PATH.stat().st_size / (1024**2), 2))

Feature dataset saved: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\data\processed\transactions_features.csv.gz
Feature report saved: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\engineered_feature_report.csv
Final shape: (6362620, 22)
Compressed file size (MB): 297.79


In [11]:
print("NOTEBOOK 03 — FEATURE ENGINEERING SUMMARY")
print("-" * 45)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Fraud rows:", int((df["isFraud"] == 1).sum()))
print("Normal rows:", int((df["isFraud"] == 0).sum()))

print("\nRequired engineered features:")
for feature in engineered_features:
    print(
        f"{feature}: "
        f"exists={feature in df.columns}, "
        f"missing={int(df[feature].isna().sum()) if feature in df.columns else 'N/A'}"
    )

print("\nPotential leakage field present but not approved as a feature:",
      "isFlaggedFraud" in df.columns)

print("\nSaved dataset exists:", FEATURES_PATH.exists())
print("Saved feature report exists:",
      (OUTPUT_DIR / "engineered_feature_report.csv").exists())

NOTEBOOK 03 — FEATURE ENGINEERING SUMMARY
---------------------------------------------
Rows: 6362620
Columns: 22
Fraud rows: 8213
Normal rows: 6354407

Required engineered features:
origin_balance_error: exists=True, missing=0
destination_balance_error: exists=True, missing=0
origin_balance_error_abs: exists=True, missing=0
destination_balance_error_abs: exists=True, missing=0
origin_emptied: exists=True, missing=0
destination_balance_zero: exists=True, missing=0
hour_of_day: exists=True, missing=0
night_flag: exists=True, missing=0
amount_to_origin_balance_ratio: exists=True, missing=0
origin_balance_was_zero: exists=True, missing=0
origin_txn_count_prior_steps: exists=True, missing=0

Potential leakage field present but not approved as a feature: True

Saved dataset exists: True
Saved feature report exists: True
